In [1]:
import numpy as np
from scipy import stats
import pandas as pd

# 1. Исходные данные
group_a = np.array([51.99, 49.45, 52.59, 56.09, 49.06, 49.06, 56.32, 53.07, 48.12, 52.17, 48.15, 48.14, 50.97, 42.35, 43.10])
group_b = np.array([48.63, 45.92, 53.89, 46.55, 43.53, 60.79, 50.65, 52.41, 43.45, 48.73])
alpha = 0.05

n1 = len(group_a)
n2 = len(group_b)

print(f"Размер выборки A (n1): {n1}")
print(f"Размер выборки B (n2): {n2}")
print("-" * 50)

# Описательные статистики
mean_a = np.mean(group_a)
var_a = np.var(group_a, ddof=1) # Выборочная дисперсия (ddof=1)

mean_b = np.mean(group_b)
var_b = np.var(group_b, ddof=1) # Выборочная дисперсия (ddof=1)

print(f"Среднее A: {mean_a:.3f}, Дисперсия A: {var_a:.3f}")
print(f"Среднее B: {mean_b:.3f}, Дисперсия B: {var_b:.3f}")
print("-" * 50)

# 2. Проверка равенства дисперсий

# F-test (scipy.stats.f_oneway можно использовать для сравнения дисперсий, но проще вручную или через функцию)
# Используем scipy.stats.bartlett или manual F-test logic.
# F-test ручной расчет:
F_stat_manual = var_b / var_a if var_b >= var_a else var_a / var_b
# P-value для F-теста обычно требует двустороннего подхода
# Scipy предоставляет более удобные функции для проверки равенства дисперсий.

# Levene test
levene_stat, levene_pvalue = stats.levene(group_a, group_b)

print(f"Levene's test statistic: {levene_stat:.3f}, p-value: {levene_pvalue:.3f}")

if levene_pvalue < alpha:
    print("Результат Levene's test: Отвергаем H0 о равенстве дисперсий. Дисперсии не равны (p < alpha).")
    equal_variances = False
else:
    print("Результат Levene's test: Не отвергаем H0 о равенстве дисперсий. Принимаем равные дисперсии (p > alpha).")
    equal_variances = True

print("-" * 50)

# 3. Выполнение t-test (pooled t-test или Welch t-test)
# Функция stats.ttest_ind имеет параметр equal_var

if equal_variances:
    print("Используется Pooled t-test (equal_var=True)")
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=True)
else:
    print("Используется Welch's t-test (equal_var=False)")
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)

print(f"T-statistic: {t_stat:.3f}, P-value: {p_value:.3f}")

if p_value < alpha:
    print("Результат t-теста: Отвергаем H0: Разница в средних статистически значима (p < alpha).")
else:
    print("Результат t-теста: Не отвергаем H0: Разница в средних статистически незначима (p > alpha).")

print("-" * 50)

# 4. Построение 95% доверительного интервала для разности средних

# Разность средних
diff_means = mean_a - mean_b

if equal_variances:
    # Pooled Standard Error и Degrees of Freedom
    pooled_var = ((n1 - 1) * var_a + (n2 - 1) * var_b) / (n1 + n2 - 2)
    std_error_diff = np.sqrt(pooled_var * (1/n1 + 1/n2))
    df = n1 + n2 - 2
else:
    # Welch Standard Error и Degrees of Freedom (сложная формула Уэлча, используем приближение)
    # Scipy ttest_ind возвращает только p-value и stat, для CI нужно считать df Welch'а
    # Примем, что Welch df можно получить по формуле или используем библиотеку statsmodels для простоты CI
    # df для Welch можно оценить так:
    # Scipy internally calculates correct df for Welch test. 
    # Для ручного расчета df для Welch's:
    # df_welch = ( (s1^2/n1 + s2^2/n2)^2 ) / ( (s1^2/n1)^2/(n1-1) + (s2^2/n2)^2/(n2-1) )
    se_a = var_a / n1
    se_b = var_b / n2
    std_error_diff = np.sqrt(se_a + se_b)
    df = (se_a + se_b)**2 / (se_a**2 / (n1 - 1) + se_b**2 / (n2 - 1))
    
    
# Критическое значение t для 95% CI (alpha=0.05, двусторонний)
t_critical = stats.t.ppf(1 - alpha/2, df)

margin_of_error = t_critical * std_error_diff
ci_lower = diff_means - margin_of_error
ci_upper = diff_means + margin_of_error

print(f"95% Доверительный интервал для разности средних (A - B):")
print(f"[{ci_lower:.3f} MPa; {ci_upper:.3f} MPa]")

if ci_lower < 0 < ci_upper:
    print("Интерпретация CI: Интервал содержит ноль, подтверждая отсутствие стат. значимой разницы.")
else:
    print("Интерпретация CI: Интервал не содержит ноль, подтверждая наличие стат. значимой разницы.")


Размер выборки A (n1): 15
Размер выборки B (n2): 10
--------------------------------------------------
Среднее A: 50.042, Дисперсия A: 15.812
Среднее B: 49.455, Дисперсия B: 27.958
--------------------------------------------------
Levene's test statistic: 0.468, p-value: 0.501
Результат Levene's test: Не отвергаем H0 о равенстве дисперсий. Принимаем равные дисперсии (p > alpha).
--------------------------------------------------
Используется Pooled t-test (equal_var=True)
T-statistic: 0.317, P-value: 0.754
Результат t-теста: Не отвергаем H0: Разница в средних статистически незначима (p > alpha).
--------------------------------------------------
95% Доверительный интервал для разности средних (A - B):
[-3.243 MPa; 4.417 MPa]
Интерпретация CI: Интервал содержит ноль, подтверждая отсутствие стат. значимой разницы.


In [2]:
import numpy as np
from scipy import stats
from pingouin import ttest

# 1. Исходные данные
staraia_shema = np.array([12.5, 13.1, 11.8, 12.9, 13.5, 12.0, 13.2])
novaia_shema = np.array([12.2, 13.0, 12.0, 12.7, 13.6, 11.9, 13.4])

# Расчет разностей (Новое - Старое)
differentials = novaia_shema - staraia_shema
print(f"Разности (d): {differentials}\n")

# --- а) Проверка нормальности распределения разностей (Shapiro-Wilk) ---
# H0: Распределение нормальное
# H1: Распределение ненормальное

shapiro_test_statistic, shapiro_p_value = stats.shapiro(differentials)

print("--- а) Проверка нормальности (Shapiro-Wilk) ---")
print(f"Статистика критерия W: {shapiro_test_statistic:.3f}")
print(f"P-value: {shapiro_p_value:.3f}")

if shapiro_p_value > 0.05:
    print("Вывод: P-value > 0.05. Нет оснований отвергать H0. Распределение можно считать нормальным.")
    use_parametric = True
else:
    print("Вывод: P-value <= 0.05. H0 отвергается. Распределение не является нормальным.")
    use_parametric = False

print("\n")

# --- б) Выполнение парного теста (t-test или Wilcoxon) ---

print("--- б) Парный тест (t-test или Wilcoxon) ---")
if use_parametric:
    # Парный t-test
    # Используем библиотеку pingouin для удобства получения CI и Cohen's d
    # Параметр paired=True указывает на парный тест
    # Используем ttest из pingouin, так как он возвращает больше метрик сразу
    
    t_result = ttest(x=novaia_shema, y=staraia_shema, paired=True, alternative='two-sided')
    print("Использован парный t-критерий:")
    print(t_result)
    p_val_ttest = t_result['p-val'].values[0]

    if p_val_ttest < 0.05:
        print(f"\nВывод: P-value ({p_val_ttest:.3f}) <= 0.05. Есть статистически значимое изменение (отвергаем H0).")
    else:
        print(f"\nВывод: P-value ({p_val_ttest:.3f}) > 0.05. Нет статистически значимого изменения (принимаем H0).")

else:
    # Непараметрический критерий Уилкоксона для парных выборок
    wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(x=staraia_shema, y=novaia_shema)
    print("Использован критерий Уилкоксона:")
    print(f"Статистика критерия W: {wilcoxon_statistic:.3f}")
    print(f"P-value: {wilcoxon_p_value:.3f}")
    
    if wilcoxon_p_value < 0.05:
        print(f"\nВывод: P-value ({wilcoxon_p_value:.3f}) <= 0.05. Есть статистически значимое изменение.")
    else:
        print(f"\nВывод: P-value ({wilcoxon_p_value:.3f}) > 0.05. Нет статистически значимого изменения.")

print("\n")

# --- в) Доверительный интервал и d Коэна (используя результаты ttest из pingouin) ---

print("--- в) Доверительный интервал и d Коэна ---")
if use_parametric:
    ci_lower = t_result['CI95%'].values[0][0]
    ci_upper = t_result['CI95%'].values[0][1]
    cohens_d = t_result['cohen-d'].values[0]
    
    print(f"95% Доверительный интервал (CI): [{ci_lower:.3f}, {ci_upper:.3f}]")
    print(f"Парный d Коэна: {cohens_d:.3f}")
    print(f"Размер эффекта: {'Малый' if abs(cohens_d) < 0.5 else ('Средний' if abs(cohens_d) < 0.8 else 'Большой')}")



Разности (d): [-0.3 -0.1  0.2 -0.2  0.1 -0.1  0.2]

--- а) Проверка нормальности (Shapiro-Wilk) ---
Статистика критерия W: 0.907
P-value: 0.376
Вывод: P-value > 0.05. Нет оснований отвергать H0. Распределение можно считать нормальным.


--- б) Парный тест (t-test или Wilcoxon) ---
Использован парный t-критерий:
               T  dof alternative     p-val          CI95%   cohen-d   BF10  \
T-test -0.382546    6   two-sided  0.715241  [-0.21, 0.15]  0.043415  0.375   

           power  
T-test  0.051097  

Вывод: P-value (0.715) > 0.05. Нет статистически значимого изменения (принимаем H0).


--- в) Доверительный интервал и d Коэна ---
95% Доверительный интервал (CI): [-0.210, 0.150]
Парный d Коэна: 0.043
Размер эффекта: Малый
